# LZ78 Complexity Sentiment Classification Experiment

## Overview

This notebook implements and evaluates LZ78 complexity-based sentiment classification. 
The experiment compares the LZ78 approach against baseline methods (VADER, TextBlob, NPC-gzip) 
on sentiment analysis tasks.

### What This Notebook Does:

1. **LZ78 Complexity Computation**: Uses gzip compressed length as a proxy for LZ78 complexity
2. **Statistical Analysis**: Mann-Whitney U test and Cliff's delta effect size
3. **Theoretical Threshold**: Derives classification threshold from information-theoretic principles
4. **Direction-Aware Classifier**: Learns whether lower or higher complexity indicates positive sentiment
5. **Baseline Comparisons**: Compares against VADER, TextBlob, and NPC-gzip
6. **Visualization**: Plots complexity distributions and accuracy comparisons

### Key Findings (from full experiment):
- LZ78 complexity shows statistically significant but modest correlation with sentiment for Amazon reviews (p<0.001, accuracy 53.2%) and tweets (p<0.001, accuracy 62.9%)
- No significant correlation for IMDB reviews (p=0.316, accuracy ~50% random)
- Baselines (VADER, TextBlob) achieve 60-75% accuracy
- LZ78 complexity captures sentiment in short texts but is less informative for longer reviews

In [ ]:
# Install dependencies - works on both Colab and local Jupyter
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Packages NOT pre-installed on Colab (always install everywhere)
_pip('loguru')
_pip('nltk')
_pip('textblob')
_pip('scikit-learn')
_pip('seaborn')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scipy==1.16.3', 'matplotlib==3.10.0')

In [ ]:
# Standard imports (copied from method.py with minimal additions)
from loguru import logger
from pathlib import Path
import json
import sys
import zlib
import numpy as np
import scipy.stats as stats
from collections import Counter
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from textblob import TextBlob
import gc
import time
from typing import List, Dict, Tuple, Any

# Configure logging (simplified for notebook)
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

# Download NLTK data (needed for VADER)
import nltk
nltk.download('vader_lexicon', quiet=True)
nltk.download('punkt', quiet=True)

In [ ]:
# Data loading helper - GitHub URL with local fallback
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-a81b4f-lz78-complexity-for-rule-based-sentiment/main/round-2/experiment-1/demo/mini_demo_data.json"

import json, os

def load_data():
    """Load demo data from GitHub URL with local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

# Test the loading
print("Data loading function defined with GitHub URL:")
print(GITHUB_DATA_URL)

In [ ]:
# Load the demo data
data = load_data()

print("Data loaded successfully!")
print(f"Train examples: {len(data['train'])}")
print(f"Test examples: {len(data['test'])}")
print(f"\nSample train example:")
print(f"  Text: {data['train'][0]['text'][:80]}...")
print(f"  Label: {data['train'][0]['label']}")

## Configuration

Set all tunable parameters here. For the demo, we use ABSOLUTE MINIMUM values 
to ensure the notebook runs quickly. To run a more comprehensive experiment, 
increase these values gradually.

In [ ]:
# Configuration - ABSOLUTE MINIMUM values for demo
# Increase these gradually for more comprehensive results

# Number of training examples to use (None = all)
MAX_TRAIN_EXAMPLES = 10  # Minimum: 10 (5 per class for meaningful stats)

# Number of test examples to use (None = all)
MAX_TEST_EXAMPLES = 10   # Minimum: 10 (5 per class for meaningful evaluation)

# k for NPC-gzip (number of neighbors)
NPC_K = 1  # Minimum: 1 (for speed)

# NPC-gzip training sample size (for speed)
NPC_TRAIN_SAMPLE = 10  # Minimum: 10 (5 per class)

print("Configuration set:")
print(f"  MAX_TRAIN_EXAMPLES = {MAX_TRAIN_EXAMPLES}")
print(f"  MAX_TEST_EXAMPLES = {MAX_TEST_EXAMPLES}")
print(f"  NPC_K = {NPC_K}")
print(f"  NPC_TRAIN_SAMPLE = {NPC_TRAIN_SAMPLE}")

## LZ78 Complexity Computation

LZ78 complexity is computed using gzip compressed length as a proxy (based on Jiang et al. 2023 ACL Findings).
The complexity is normalized by text length to allow fair comparison across texts of different lengths.

**Theory**: LZ78 normalized complexity converges to the entropy rate H(X). Texts with more 
repetitive patterns (lower entropy) will have lower complexity scores.

In [ ]:
def lz78_complexity(text: str, normalize: bool = True) -> float:
    """LZ78 complexity using gzip compressed length as proxy.
    
    Based on Jiang et al. (2023) ACL Findings.
    
    Args:
        text: Input text
        normalize: Whether to normalize by text length
    
    Returns:
        Complexity value
    """
    if not text or len(text) == 0:
        return 0.0
    
    text_bytes = text.encode('utf-8')
    compressed = zlib.compress(text_bytes, level=9)
    complexity = len(compressed)
    
    if normalize:
        complexity = complexity / len(text_bytes)
    
    return complexity

# Test the function
test_texts = ["hello world", "hello hello hello", "ababababab"]
print("Testing LZ78 complexity computation:")
for t in test_texts:
    c = lz78_complexity(t)
    print(f"  '{t[:20]}...' -> complexity = {c:.4f}")

## Statistical Analysis: Testing Complexity Differences

We use the **Mann-Whitney U test** to determine if there's a statistically significant 
difference in LZ78 complexity between positive and negative sentiment classes.

**Cliff's delta** is used as an effect size measure:
- |δ| < 0.147: negligible effect
- 0.147 ≤ |δ| < 0.33: small effect
- 0.33 ≤ |δ| < 0.474: medium effect
- |δ| ≥ 0.474: large effect

In [ ]:
def test_complexity_difference(pos_complexities: List[float], neg_complexities: List[float]) -> Dict[str, Any]:
    """Mann-Whitney U test + Cliff's delta effect size.
    
    Args:
        pos_complexities: List of complexities for positive examples
        neg_complexities: List of complexities for negative examples
    
    Returns:
        Dictionary with test results
    """
    statistic, p_value = stats.mannwhitneyu(
        pos_complexities, neg_complexities, alternative='two-sided'
    )
    
    # Cliff's delta
    n_pos, n_neg = len(pos_complexities), len(neg_complexities)
    greater = sum(a > b for a in pos_complexities for b in neg_complexities)
    less = sum(a < b for a in pos_complexities for b in neg_complexities)
    cliffs_delta = (greater - less) / (n_pos * n_neg)
    
    return {
        'statistic': float(statistic),
        'p_value': float(p_value),
        'cliffs_delta': float(cliffs_delta),
        'significant': bool(p_value < 0.05),
        'mean_pos': float(np.mean(pos_complexities)),
        'mean_neg': float(np.mean(neg_complexities)),
        'std_pos': float(np.std(pos_complexities)),
        'std_neg': float(np.std(neg_complexities))
    }

print("Statistical test function defined.")

## Theoretical Threshold Derivation

The threshold is derived from information-theoretic first principles.

**Theory**: LZ78 normalized complexity converges to the entropy rate H(X). 
For equal priors P(pos)=P(neg)=0.5, the optimal Bayes threshold 
under symmetric loss is the **midpoint between class-conditional means**.

The classifier also learns the **direction** - whether lower or higher 
complexity indicates positive sentiment (dataset-dependent).

In [ ]:
def derive_threshold(pos_complexities: List[float], neg_complexities: List[float]) -> float:
    """Derive threshold from information-theoretic first principles.
    
    Theory: LZ78 normalized complexity converges to entropy rate H(X).
    For equal priors P(pos)=P(neg)=0.5, optimal Bayes threshold
    under symmetric loss is midpoint between class-conditional means.
    
    Args:
        pos_complexities: List of complexities for positive examples
        neg_complexities: List of complexities for negative examples
    
    Returns:
        Threshold value
    """
    mean_pos = np.mean(pos_complexities)
    mean_neg = np.mean(neg_complexities)
    # Midpoint threshold (theoretically justified by equal priors)
    threshold = (mean_pos + mean_neg) / 2
    return float(threshold)

class LZ78SentimentClassifier:
    """LZ78-based sentiment classifier using complexity threshold."""
    
    def __init__(self, threshold: float = None, direction: str = None):
        self.threshold = threshold
        self.direction = direction  # 'lower' if lower complexity = positive, 'higher' if higher = positive
    
    def fit(self, pos_complexities: List[float], neg_complexities: List[float]):
        """Derive threshold and direction from data.
        
        The direction is determined by which class has lower mean complexity.
        """
        mean_pos = np.mean(pos_complexities)
        mean_neg = np.mean(neg_complexities)
        
        # Determine direction
        if mean_pos < mean_neg:
            self.direction = 'lower'  # Lower complexity = positive
        else:
            self.direction = 'higher'  # Higher complexity = positive
        
        # Derive threshold (midpoint between means)
        self.threshold = (mean_pos + mean_neg) / 2
        logger.info(f"Derived threshold: {self.threshold:.4f}, direction: {self.direction}")
    
    def predict(self, texts: List[str]) -> List[str]:
        """Predict sentiment for texts."""
        if self.threshold is None or self.direction is None:
            raise ValueError("Threshold or direction not set. Call fit() first.")
        
        predictions = []
        for text in texts:
            c = lz78_complexity(text)
            if self.direction == 'lower':
                pred = 'positive' if c < self.threshold else 'negative'
            else:  # direction == 'higher'
                pred = 'positive' if c > self.threshold else 'negative'
            predictions.append(pred)
        return predictions

print("Threshold derivation and LZ78 classifier defined.")

## Baseline Methods

We compare LZ78 complexity against three baseline sentiment classifiers:

1. **VADER** (Valence Aware Dictionary and sEntiment Reasoner): 
   Rule-based sentiment analysis tool specifically attuned to social media texts

2. **TextBlob**: Simple rule-based sentiment analysis using a pre-trained lexicon

3. **NPC-gzip** (Normalized Compression Distance + kNN): 
   Non-parametric method using gzip compression as a distance measure

In [ ]:
def vader_predict(texts: List[str]) -> List[str]:
    """VADER sentiment prediction."""
    analyzer = SentimentIntensityAnalyzer()
    return ['positive' if analyzer.polarity_scores(t)['compound'] >= 0 else 'negative'
            for t in texts]

def textblob_predict(texts: List[str]) -> List[str]:
    """TextBlob sentiment prediction."""
    return ['positive' if TextBlob(t).sentiment.polarity >= 0 else 'negative'
            for t in texts]

def npc_gzip_predict(train_texts: List[str], train_labels: List[str],
                     test_texts: List[str], k: int = 3) -> List[str]:
    """kNN + Normalized Compression Distance using gzip.
    
    Optimized version using compressed length directly instead of full NCD.
    
    Args:
        train_texts: Training text examples
        train_labels: Training labels
        test_texts: Test text examples
        k: Number of neighbors
    
    Returns:
        List of predictions
    """
    def gzip_len(text: str) -> int:
        return len(zlib.compress(text.encode('utf-8'), level=9))
    
    # Pre-compute training compressed lengths
    train_lens = [gzip_len(t) for t in train_texts]
    
    predictions = []
    for i, test_text in enumerate(test_texts):
        if i % 100 == 0:
            logger.info(f"NPC-gzip: processing example {i}/{len(test_texts)}")
        
        test_len = gzip_len(test_text)
        
        # Compute distances based on compressed length difference
        distances = [(abs(test_len - tl), lbl) for tl, lbl in zip(train_lens, train_labels)]
        distances.sort(key=lambda x: x[0])
        
        # Vote
        votes = Counter(lbl for _, lbl in distances[:k])
        predictions.append(votes.most_common(1)[0][0])
    
    return predictions

print("Baseline methods defined.")

In [ ]:
def evaluate(true_labels: List[str], pred_labels: List[str]) -> Dict[str, float]:
    """Evaluate predictions.
    
    Args:
        true_labels: Ground truth labels
        pred_labels: Predicted labels
    
    Returns:
        Dictionary with evaluation metrics
    """
    return {
        'accuracy': float(accuracy_score(true_labels, pred_labels)),
        'precision_pos': float(precision_score(true_labels, pred_labels, pos_label='positive')),
        'recall_pos': float(recall_score(true_labels, pred_labels, pos_label='positive')),
        'f1_pos': float(f1_score(true_labels, pred_labels, pos_label='positive')),
        'f1_macro': float(f1_score(true_labels, pred_labels, average='macro')),
        'confusion_matrix': confusion_matrix(true_labels, pred_labels,
                                           labels=['positive', 'negative']).tolist()
    }

print("Evaluation function defined.")

## Running the Experiment

Now we run the main experiment:
1. Prepare train/test data from the loaded dataset
2. Compute LZ78 complexities for all texts
3. Perform statistical test (Mann-Whitney U)
4. Fit LZ78 classifier (derive threshold and direction)
5. Run all classifiers (LZ78, VADER, TextBlob, NPC-gzip)
6. Evaluate and compare results

In [ ]:
# Prepare data from loaded demo data
print("="*60)
print("RUNNING LZ78 SENTIMENT CLASSIFICATION EXPERIMENT")
print("="*60)

# Use config variables
train_data = data['train'][:MAX_TRAIN_EXAMPLES]
test_data = data['test'][:MAX_TEST_EXAMPLES]

train_texts = [item['text'] for item in train_data]
train_labels = [item['label'] for item in train_data]
test_texts = [item['text'] for item in test_data]
test_labels = [item['label'] for item in test_data]

print(f"Train examples: {len(train_texts)}")
print(f"Test examples: {len(test_texts)}")
print(f"Train label distribution: {Counter(train_labels)}")
print(f"Test label distribution: {Counter(test_labels)}")

# Compute LZ78 complexities for training data
print("\nComputing LZ78 complexities for training examples...")
start_time = time.time()
train_complexities = [lz78_complexity(t) for t in train_texts]
print(f"Computed in {time.time() - start_time:.2f}s")

# Separate training complexities by class
train_pos_c = [c for c, l in zip(train_complexities, train_labels) if l == 'positive']
train_neg_c = [c for c, l in zip(train_complexities, train_labels) if l == 'negative']

print(f"\nTrain complexity stats:")
print(f"  Positive: mean={np.mean(train_pos_c):.4f}, std={np.std(train_pos_c):.4f}")
print(f"  Negative: mean={np.mean(train_neg_c):.4f}, std={np.std(train_neg_c):.4f}")

# Compute LZ78 complexities for test data
print("\nComputing LZ78 complexities for test examples...")
start_time = time.time()
test_complexities = [lz78_complexity(t) for t in test_texts]
print(f"Computed in {time.time() - start_time:.2f}s")

# Separate test complexities by class (for statistical test)
test_pos_c = [c for c, l in zip(test_complexities, test_labels) if l == 'positive']
test_neg_c = [c for c, l in zip(test_complexities, test_labels) if l == 'negative']

# RQ1: Statistical test on test data
print("\n" + "="*60)
print("STATISTICAL TEST (Mann-Whitney U)")
print("="*60)
stats_results = test_complexity_difference(test_pos_c, test_neg_c)
print(f"U statistic: {stats_results['statistic']:.2f}")
print(f"p-value: {stats_results['p_value']:.6f}")
print(f"Significant (p<0.05): {stats_results['significant']}")
print(f"Cliff's delta: {stats_results['cliffs_delta']:.4f}")
print(f"Mean positive: {stats_results['mean_pos']:.4f}")
print(f"Mean negative: {stats_results['mean_neg']:.4f}")

# Fit LZ78 classifier on training data
print("\n" + "="*60)
print("FITTING LZ78 CLASSIFIER")
print("="*60)
lz78_classifier = LZ78SentimentClassifier()
lz78_classifier.fit(train_pos_c, train_neg_c)

# Classify with LZ78
print("\n" + "="*60)
print("CLASSIFICATION RESULTS")
print("="*60)
print("Running LZ78 classifier...")
lz78_preds = lz78_classifier.predict(test_texts)
lz78_eval = evaluate(test_labels, lz78_preds)
print(f"LZ78 Accuracy: {lz78_eval['accuracy']:.4f}")
print(f"LZ78 F1 (macro): {lz78_eval['f1_macro']:.4f}")
print(f"Confusion Matrix: {lz78_eval['confusion_matrix']}")

# Baselines
print("\nRunning VADER baseline...")
vader_start = time.time()
vader_preds = vader_predict(test_texts)
vader_eval = evaluate(test_labels, vader_preds)
print(f"VADER Accuracy: {vader_eval['accuracy']:.4f} (time: {time.time()-vader_start:.2f}s)")

print("\nRunning TextBlob baseline...")
tb_start = time.time()
tb_preds = textblob_predict(test_texts)
tb_eval = evaluate(test_labels, tb_preds)
print(f"TextBlob Accuracy: {tb_eval['accuracy']:.4f} (time: {time.time()-tb_start:.2f}s)")

# NPC-gzip baseline (with reduced training sample for speed)
print(f"\nRunning NPC-gzip baseline (with {NPC_TRAIN_SAMPLE} training examples)...")
npc_start = time.time()
npc_train_sample = train_data[:NPC_TRAIN_SAMPLE]
npc_train_texts = [item['text'] for item in npc_train_sample]
npc_train_labels = [item['label'] for item in npc_train_sample]
npc_preds = npc_gzip_predict(npc_train_texts, npc_train_labels, test_texts, k=NPC_K)
npc_eval = evaluate(test_labels, npc_preds)
print(f"NPC-gzip Accuracy: {npc_eval['accuracy']:.4f} (time: {time.time()-npc_start:.2f}s)")

# Store results
results = {
    'lz78': lz78_eval,
    'vader': vader_eval,
    'textblob': tb_eval,
    'npc_gzip': npc_eval
}

print("\n" + "="*60)
print("EXPERIMENT COMPLETED!")
print("="*60)

## Visualization: Complexity Distributions

Visualize the distribution of LZ78 complexity values for positive vs negative examples.
This helps understand whether complexity differs systematically between sentiment classes.

In [ ]:
def plot_distributions(pos_c: List[float], neg_c: List[float]):
    """Plot complexity distributions for positive and negative examples."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # Boxplot
    bp = ax1.boxplot([pos_c, neg_c])
    ax1.set_xticks([1, 2])
    ax1.set_xticklabels(['Positive', 'Negative'])
    ax1.set_title('Complexity Distribution (Boxplot)')
    ax1.set_ylabel('LZ78 Complexity (normalized)')
    
    # Violin plot
    vp = sns.violinplot(data=[pos_c, neg_c], ax=ax2)
    ax2.set_xticks([0, 1])
    ax2.set_xticklabels(['Positive', 'Negative'])
    ax2.set_title('Complexity Distribution (Violin)')
    ax2.set_ylabel('LZ78 Complexity (normalized)')
    
    plt.tight_layout()
    plt.show()

# Plot the distributions for test data
print("Plotting complexity distributions...")
plot_distributions(test_pos_c, test_neg_c)

## Visualization: Accuracy Comparison

Compare the accuracy of LZ78 complexity classifier against baseline methods.
Bar chart shows relative performance of each approach.

In [ ]:
def plot_accuracy_comparison(results: Dict):
    """Plot accuracy comparison across methods."""
    methods = ['lz78', 'vader', 'textblob', 'npc_gzip']
    accuracies = [results[m]['accuracy'] for m in methods]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    x = np.arange(len(methods))
    bars = ax.bar(x, accuracies, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
    
    ax.set_xlabel('Method')
    ax.set_ylabel('Accuracy')
    ax.set_title('Sentiment Classification Accuracy Comparison')
    ax.set_xticks(x)
    ax.set_xticklabels([m.upper() for m in methods])
    ax.set_ylim([0, 1])
    
    # Add value labels on bars
    for bar, acc in zip(bars, accuracies):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{acc:.3f}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()

# Plot accuracy comparison
print("Plotting accuracy comparison...")
plot_accuracy_comparison(results)

## Results Summary

Summary of experiment results comparing LZ78 complexity-based classification 
against baseline methods.

In [ ]:
# Print comprehensive results table
print("="*70)
print("RESULTS SUMMARY")
print("="*70)
print()

# Statistical test results
print("Statistical Test (Mann-Whitney U):")
print(f"  U statistic: {stats_results['statistic']:.2f}")
print(f"  p-value: {stats_results['p_value']:.6f}")
print(f"  Significant (p<0.05): {stats_results['significant']}")
print(f"  Cliff's delta: {stats_results['cliffs_delta']:.4f}")
print()

# Classifier threshold info
print("LZ78 Classifier:")
print(f"  Derived threshold: {lz78_classifier.threshold:.4f}")
print(f"  Direction: {lz78_classifier.direction} complexity = positive")
print()

# Accuracy comparison table
print("Accuracy Comparison:")
print("-"*50)
print(f"{'Method':<15} {'Accuracy':<12} {'F1 (macro)':<12} {'Precision':<12}")
print("-"*50)
for method in ['lz78', 'vader', 'textblob', 'npc_gzip']:
    eval_result = results[method]
    print(f"{method.upper():<15} {eval_result['accuracy']:<12.4f} {eval_result['f1_macro']:<12.4f} {eval_result['precision_pos']:<12.4f}")
print("-"*50)
print()

# Interpretation
print("Interpretation:")
if stats_results['significant']:
    print("  ✓ LZ78 complexity differs significantly between sentiment classes")
else:
    print("  ✗ LZ78 complexity does NOT differ significantly between sentiment classes")
    
best_method = max(results.keys(), key=lambda m: results[m]['accuracy'])
print(f"  Best performing method: {best_method.upper()} (accuracy={results[best_method]['accuracy']:.4f})")
print()
print("="*70)